# 🏠 NhaDat Chatbot - PDF Trainer
**Đọc PDF từ Google Drive → Gemini Vision OCR → Push GitHub → RAG index**

### Chỉ cần làm 2 việc trước khi chạy:
1. Vào **Secrets (🔑 biểu tượng chìa khóa bên trái)** → thêm `GEMINI_API_KEY` và `GITHUB_TOKEN`
2. **Runtime → Run all** — xong!

In [ ]:
# CELL 1: Cài thư viện
!pip install -q pymupdf pdfplumber Pillow requests google-api-python-client google-auth
print('✅ Xong')

In [ ]:
# CELL 2: CẤU HÌNH (không cần sửa gì)
import os, base64, time, requests, io
from pathlib import Path
from google.colab import auth, userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.auth import default

# === Đã cấu hình sẵn, không cần sửa ===
DRIVE_FOLDER_ID = '18AYPZ30KCRDEm2a7p2TWgHHUKW2r7v-W'
GITHUB_OWNER    = 'quang507'
GITHUB_REPO     = 'NhaDat-chatbot'
GITHUB_BRANCH   = 'main'
DOWNLOAD_DIR    = '/content/pdfs'
OUTPUT_DIR      = '/content/output_md'

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GITHUB_TOKEN   = userdata.get('GITHUB_TOKEN')

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Xác thực Google Drive
auth.authenticate_user()
creds, _ = default()
drive_service = build('drive', 'v3', credentials=creds)

print(f'🔑 Gemini: {"✅" if GEMINI_API_KEY else "❌ CHƯA SET"}')
print(f'🔑 GitHub: {"✅" if GITHUB_TOKEN else "❌ CHƯA SET"}')
print('✅ Đã kết nối Google Drive')

In [ ]:
# CELL 3: Tải PDF từ Drive
def list_pdfs_recursive(folder_id, service):
    results = []
    page_token = None
    while True:
        resp = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields='nextPageToken, files(id, name, mimeType)',
            pageToken=page_token
        ).execute()
        for f in resp.get('files', []):
            if f['mimeType'] == 'application/vnd.google-apps.folder':
                results.extend(list_pdfs_recursive(f['id'], service))
            elif f['name'].lower().endswith('.pdf'):
                results.append(f)
        page_token = resp.get('nextPageToken')
        if not page_token: break
    return results

def download_file(file_id, dest_path, service):
    req = service.files().get_media(fileId=file_id)
    with open(dest_path, 'wb') as f:
        dl = MediaIoBaseDownload(f, req)
        done = False
        while not done: _, done = dl.next_chunk()

pdf_list = list_pdfs_recursive(DRIVE_FOLDER_ID, drive_service)
print(f'🔍 Tìm thấy {len(pdf_list)} file PDF')

local_pdfs = []
for f in pdf_list:
    dest = os.path.join(DOWNLOAD_DIR, f['name'])
    print(f'  ⬇️ Tải: {f["name"]}', end=' ')
    download_file(f['id'], dest, drive_service)
    local_pdfs.append(dest)
    print('✅')
print(f'📥 Đã tải {len(local_pdfs)} files')

In [ ]:
# CELL 4: Hàm OCR + Extract
import fitz, pdfplumber

GEMINI_URL = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}'
PROMPT = '''Bạn là chuyên gia trích xuất dữ liệu bất động sản.
Đọc hình ảnh trang PDF và chuyển TOÀN BỘ nội dung thành Markdown có cấu trúc.
- Giữ NGUYÊN mọi số liệu: diện tích, giá, mã căn/lô, kích thước, tỉ lệ thanh toán
- Bảng → Markdown table | Tiêu đề → ## | Danh sách → bullet
- Sơ đồ/mặt bằng: liệt kê từng lô/căn với số hiệu và diện tích
- KHÔNG thêm nhận xét, chỉ chuyển dữ liệu
Trả về Markdown thuần, không dùng code block.'''

def page_to_png(page, dpi=200):
    pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72), colorspace=fitz.csRGB)
    return pix.tobytes('png')

def gemini_ocr(img_bytes, retry=3):
    b64 = base64.b64encode(img_bytes).decode()
    payload = {'contents': [{'parts': [{'text': PROMPT}, {'inline_data': {'mime_type': 'image/png', 'data': b64}}]}],
               'generationConfig': {'temperature': 0.1, 'maxOutputTokens': 8192}}
    for i in range(retry):
        r = requests.post(GEMINI_URL, json=payload, timeout=90)
        if r.status_code == 429:
            w = (i+1)*20; print(f'  ⏳ Rate limit {w}s...'); time.sleep(w); continue
        r.raise_for_status()
        return r.json()['candidates'][0]['content']['parts'][0]['text']
    return ''

def extract_text_page(pdf_path, page_num):
    with pdfplumber.open(pdf_path) as pdf:
        p = pdf.pages[page_num]
        parts = []
        for tbl in (p.extract_tables() or []):
            rows = [[str(c or '').strip() for c in row] for row in tbl if row]
            if not rows: continue
            h = rows[0]; md = '| ' + ' | '.join(h) + ' |\n| ' + ' | '.join(['---']*len(h)) + ' |\n'
            for row in rows[1:]:
                while len(row) < len(h): row.append('')
                md += '| ' + ' | '.join(row) + ' |\n'
            parts.append(md)
        txt = p.extract_text() or ''
        if txt.strip(): parts.append(txt.strip())
        return '\n\n'.join(parts)

def process_pdf(pdf_path):
    name = Path(pdf_path).stem
    print(f'\n📄 {Path(pdf_path).name}')
    doc = fitz.open(pdf_path)
    parts = [f'# {name}']
    for i, page in enumerate(doc):
        print(f'  Trang {i+1}/{len(doc)}', end=' ')
        if len(page.get_text().strip()) >= 80:
            print('(text)', end=' ')
            txt = extract_text_page(pdf_path, i)
            if txt.strip(): parts.append(f'## Trang {i+1}\n\n{txt.strip()}')
        else:
            print('(scan→Vision)', end=' ')
            md = gemini_ocr(page_to_png(page))
            if md.strip(): parts.append(f'## Trang {i+1}\n\n{md.strip()}')
            time.sleep(2)
        print('✅')
    doc.close()
    return '\n\n---\n\n'.join(parts)

print('✅ Sẵn sàng')

In [ ]:
# CELL 5: Xử lý tất cả PDF
results = {}
for pdf_path in local_pdfs:
    try:
        md = process_pdf(pdf_path)
        fn = Path(pdf_path).stem + '.md'
        results[fn] = md
        Path(os.path.join(OUTPUT_DIR, fn)).write_text(md, encoding='utf-8')
        print(f'  💾 {fn} ({len(md):,} ký tự)')
    except Exception as e:
        print(f'  ❌ {Path(pdf_path).name}: {e}')
print(f'\n✅ Xong {len(results)}/{len(local_pdfs)} files')

In [ ]:
# CELL 6: Push lên GitHub → data/pdf-extracted/
def gh_push(repo_path, content, msg):
    url = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{repo_path}'
    hdrs = {'Authorization': f'Bearer {GITHUB_TOKEN}', 'Accept': 'application/vnd.github+json'}
    r = requests.get(f'{url}?ref={GITHUB_BRANCH}', headers=hdrs)
    sha = r.json().get('sha') if r.ok else None
    body = {'message': msg, 'branch': GITHUB_BRANCH, 'content': base64.b64encode(content.encode()).decode()}
    if sha: body['sha'] = sha
    r = requests.put(url, headers=hdrs, json=body)
    print(f'  {"✅" if r.ok else "❌"} {repo_path}')
    return r.ok

if GITHUB_TOKEN and results:
    print('🚀 Pushing lên GitHub...')
    ok = 0
    for fn, content in results.items():
        if gh_push(f'data/pdf-extracted/{fn}', content, f'data: {fn} via Colab Vision'): ok += 1
        time.sleep(1)
    print(f'\n✅ {ok}/{len(results)} files → data/pdf-extracted/')
    print('🔔 Chạy Chay_Dong_Bo.bat để rebuild RAG index!')
else:
    print('⚠️ Thiếu GITHUB_TOKEN → file đã lưu tại /content/output_md/')